# BÀI 7: TỰ ĐỘNG TÁCH FILE BÁO CÁO & XỬ LÝ DỮ LIỆU LỚN (CHUNKING)
## Mục tiêu thực hành:
1. **Tự động tách file (Batch Splitting):** Từ 1 bảng Master Data tổng hợp, tự động tách thành nhiều file Excel riêng biệt theo từng Phòng ban / Chi nhánh.
2. **Định dạng chuẩn từng file tách:** Tích hợp hàm trang trí OpenPyXL (Header thương hiệu, kẻ viền, auto-fit cột) cho tất cả các file con xuất ra.
3. **Xử lý dữ liệu siêu lớn không tràn RAM (`chunksize`):** Đọc cuốn chiếu từng gói dữ liệu (chunk) từ file CSV dung lượng lớn để lọc và tổng hợp số liệu.

### Ô 1: Chuẩn bị bảng dữ liệu tổng hợp Master Data của toàn hệ thống
> Giả lập bảng tổng hợp tài sản của 4 phòng ban/đơn vị khác nhau.

In [1]:
import pandas as pd
from pathlib import Path
import re

data_master = [
    {"Don_Vi": "Phòng Kế toán", "Ma_TS": "KT-01", "Ten_TS": "Máy tính Dell OptiPlex", "So_luong": 4, "Gia_tri": 64000000},
    {"Don_Vi": "Phòng Kế toán", "Ma_TS": "KT-02", "Ten_TS": "Máy in hóa đơn Epson", "So_luong": 2, "Gia_tri": 11000000},
    {"Don_Vi": "Phòng Tín dụng", "Ma_TS": "TD-01", "Ten_TS": "Laptop HP ProBook 450", "So_luong": 6, "Gia_tri": 114000000},
    {"Don_Vi": "Phòng Tín dụng", "Ma_TS": "TD-02", "Ten_TS": "Máy scan tài liệu Canon", "So_luong": 3, "Gia_tri": 25500000},
    {"Don_Vi": "PGD Ninh Kiều", "Ma_TS": "NK-01", "Ten_TS": "Máy đếm tiền Glory", "So_luong": 3, "Gia_tri": 18000000},
    {"Don_Vi": "PGD Cái Răng", "Ma_TS": "CR-01", "Ten_TS": "Bộ lưu điện UPS APC", "So_luong": 2, "Gia_tri": 9600000}
]

df_master = pd.DataFrame(data_master)
display(df_master)

,Don_Vi,Ma_TS,Ten_TS,So_luong,Gia_tri
0,Phòng Kế toán,KT-01,Máy tính Dell OptiPlex,4,64000000
1,Phòng Kế toán,KT-02,Máy in hóa đơn Epson,2,11000000
2,Phòng Tín dụng,TD-01,Laptop HP ProBook 450,6,114000000
3,Phòng Tín dụng,TD-02,Máy scan tài liệu Canon,3,25500000
4,PGD Ninh Kiều,NK-01,Máy đếm tiền Glory,3,18000000
5,PGD Cái Răng,CR-01,Bộ lưu điện UPS APC,2,9600000


### Ô 2: Xây dựng hàm xuất và trang trí chuẩn OpenPyXL dùng chung
> Đóng gói kỹ thuật định dạng Excel (Bài 4) thành một hàm tái sử dụng (`xuat_excel_chuan`) để áp dụng cho hàng loạt file.

In [2]:
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

def xuat_excel_chuan(df_sub, file_path, sheet_name="DanhSachTaiSan"):
    with pd.ExcelWriter(file_path, engine="openpyxl") as writer:
        df_sub.to_excel(writer, sheet_name=sheet_name, index=False)
        ws = writer.sheets[sheet_name]
        
        # Định dạng Header
        header_fill = PatternFill(start_color="00685E", end_color="00685E", fill_type="solid")
        header_font = Font(name="Arial", size=10, bold=True, color="FFFFFF")
        thin_side = Side(border_style="thin", color="BFBFBF")
        border = Border(left=thin_side, right=thin_side, top=thin_side, bottom=thin_side)
        
        ws.row_dimensions[1].height = 25
        for col_idx in range(1, len(df_sub.columns) + 1):
            cell = ws.cell(row=1, column=col_idx)
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal="center", vertical="center")
            cell.border = border
            
        # Kẻ khung & định dạng tiền tệ
        for r in range(2, len(df_sub) + 2):
            ws.row_dimensions[r].height = 20
            for c in range(1, len(df_sub.columns) + 1):
                cell = ws.cell(row=r, column=c)
                cell.border = border
                if df_sub.columns[c - 1] == "Gia_tri":
                    cell.number_format = '#,##0" VNĐ"'
                    
        # Auto-fit cột
        for col in ws.columns:
            col_letter = get_column_letter(col[0].column)
            max_len = max(len(str(cell.value or "")) for cell in col)
            ws.column_dimensions[col_letter].width = max(max_len + 4, 12)

### Ô 3: Tự động tách file theo từng đơn vị (`groupby`) và lưu vào thư mục đầu ra
> Dùng `re.sub` để làm sạch tên đơn vị trước khi đặt làm tên file (tránh lỗi ký tự đặc biệt như `/` hoặc `:`).

In [3]:
output_dir = Path("bao_cao_tach_don_vi")
output_dir.mkdir(parents=True, exist_ok=True)

for ten_don_vi, df_nhom in df_master.groupby("Don_Vi"):
    # Làm sạch tên file an toàn trên Windows
    safe_name = re.sub(r'[\\/*?:"<>|]', "_", str(ten_don_vi)).strip()
    file_out = output_dir / f"BaoCao_{safe_name}.xlsx"
    
    xuat_excel_chuan(df_nhom, file_out)
    print(f"[ĐÃ XUẤT] {file_out.name} ({len(df_nhom)} tài sản)")

print(f"\nHoàn tất tách toàn bộ file vào thư mục: {output_dir.resolve()}")

[ĐÃ XUẤT] BaoCao_PGD Cái Răng.xlsx (1 tài sản)
[ĐÃ XUẤT] BaoCao_PGD Ninh Kiều.xlsx (1 tài sản)
[ĐÃ XUẤT] BaoCao_Phòng Kế toán.xlsx (2 tài sản)
[ĐÃ XUẤT] BaoCao_Phòng Tín dụng.xlsx (2 tài sản)

Hoàn tất tách toàn bộ file vào thư mục: D:\Python_Project\Python_data_apps\Code_vidu\bao_cao_tach_don_vi


### Ô 4: Kỹ thuật xử lý dữ liệu siêu lớn không tràn RAM (`chunksize`)
> Tạo một file CSV giả lập gồm **100.000 dòng giao dịch**, sau đó dùng tham số `chunksize=20000` để đọc cuốn chiếu từng gói 20.000 dòng, lọc và tính tổng giá trị mà chỉ tốn lượng RAM cực nhỏ.

In [4]:
import numpy as np

big_csv = Path("du_lieu_giao_dich_100k.csv")

# 1. Tạo file CSV 100.000 dòng để thử nghiệm
n_rows = 100_000
df_big_sample = pd.DataFrame({
    "Ma_GD": [f"GD-{i:06d}" for i in range(1, n_rows + 1)],
    "Don_Vi": np.random.choice(["PGD Ninh Kiều", "PGD Cái Răng", "Phòng Kế toán", "Phòng Tín dụng"], size=n_rows),
    "So_tien": np.random.randint(1_000_000, 50_000_000, size=n_rows, dtype=np.int64)
})
df_big_sample.to_csv(big_csv, index=False, encoding="utf-8-sig")
print(f"Đã tạo file CSV giả lập {n_rows:,} dòng ({big_csv.stat().st_size / 1024 / 1024:.2f} MB).\n")

# 2. Đọc cuốn chiếu từng gói 20.000 dòng bằng chunksize
tong_hop_don_vi = {}
chunk_idx = 0

for chunk in pd.read_csv(big_csv, chunksize=20_000, usecols=["Don_Vi", "So_tien"]):
    chunk_idx += 1
    # Tính tổng theo đơn vị ngay trên gói nhỏ hiện tại
    nhom_tam = chunk.groupby("Don_Vi")["So_tien"].sum()
    
    for dv, tong in nhom_tam.items():
        tong_hop_don_vi[dv] = tong_hop_don_vi.get(dv, 0) + tong
        
    print(f"Đã xử lý xong gói (chunk) số {chunk_idx} - giải phóng RAM thành công.")

# 3. Chuyển kết quả tổng hợp cuối cùng thành DataFrame
df_ket_qua_lon = pd.DataFrame(
    list(tong_hop_don_vi.items()), 
    columns=["Đơn vị", "Tổng giá trị giao dịch"]
).sort_values(by="Tổng giá trị giao dịch", ascending=False)

print("\n--- KẾT QUẢ TỔNG HỢP TỪ 100.000 DÒNG ---")
display(df_ket_qua_lon)

Đã tạo file CSV giả lập 100,000 dòng (3.51 MB).

Đã xử lý xong gói (chunk) số 1 - giải phóng RAM thành công.
Đã xử lý xong gói (chunk) số 2 - giải phóng RAM thành công.
Đã xử lý xong gói (chunk) số 3 - giải phóng RAM thành công.
Đã xử lý xong gói (chunk) số 4 - giải phóng RAM thành công.
Đã xử lý xong gói (chunk) số 5 - giải phóng RAM thành công.

--- KẾT QUẢ TỔNG HỢP TỪ 100.000 DÒNG ---


,Đơn vị,Tổng giá trị giao dịch
1,PGD Ninh Kiều,643097111017
2,Phòng Kế toán,642084710568
3,Phòng Tín dụng,637768589790
0,PGD Cái Răng,636626627796
